# HTR Batch - Celorico da Beira (Gemini 3 Flash)Processa 4152 imagens de registos de óbitos via Gemini API.## 1. Instalar dependências

In [ ]:
!apt-get install -y tesseract-ocr tesseract-ocr-por > /dev/null 2>&1!pip install -q pillow urllib3

## 2. Configurar API KeyChave já configurada abaixo:

In [ ]:
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')print('Key configurada:', GEMINI_API_KEY[:20] + '...')

## 3. Fazer upload das imagens do servidorNo servidor: `cd /home/pxtkhw/projetos/obitos && tar czf full_images.tar.gz output/full_images/`Depois faz upload do `full_images.tar.gz` aqui:

In [ ]:
from google.colab import filesimport osprint('Faz upload do full_images.tar.gz...')# uploaded = files.upload()  # Descomenta para fazer uploados.makedirs('full_images', exist_ok=True)!tar xzf full_images.tar.gz -C . 2>/dev/null || echo 'Faz upload primeiro!'print('Imagens disponíveis:', len(os.listdir('full_images')))

## 4. Script HTR (processa todas as imagens)

In [ ]:
import base64, json, time, os, refrom PIL import Imagefrom io import BytesIOimport urllib.requestdef process_image(img_path):    img = Image.open(img_path).convert('RGB')    buf = BytesIO()    img.save(buf, format='JPEG', quality=85)    b64 = base64.b64encode(buf.getvalue()).decode()        prompt = '''You are a transcription assistant for Portuguese historical documents.Output ONLY a JSON object (no markdown) with this structure:{  \"transcription\": \"full transcribed text here\",  \"deceased\": [    {      \"name\": \"person name\",      \"death_date\": \"YYYY-MM-DD\",      \"age\": \"age if mentioned\"    }  ]}IMPORTANT: For death_date, ALWAYS use ISO format (YYYY-MM-DD). Output ONLY the JSON object.'''        payload = {        'contents': [{'parts': [            {'text': prompt},            {'inline_data': {'mime_type': 'image/jpeg', 'data': b64}}        ]}],        'generationConfig': {'temperature': 0.1, 'maxOutputTokens': 2000}    }        data = json.dumps(payload).encode()    url = f'https://generativelanguage.googleapis.com/v1beta/models/gemini-3-flash-preview:generateContent?key={GEMINI_API_KEY}'    req = urllib.request.Request(url, data=data, headers={'Content-Type': 'application/json'})    with urllib.request.urlopen(req, timeout=60) as resp:        result = json.loads(resp.read())        return result['candidates'][0]['content']['parts'][0]['text']os.makedirs('htr_output', exist_ok=True)images = [f for f in os.listdir('full_images') if f.endswith('.tiff')]print(f'Total imagens: {len(images)}')for i, img in enumerate(sorted(images)[:10]):  # Começa com 10 para testar    try:        result = process_image(os.path.join('full_images', img))        with open(f'htr_output/{img}.json', 'w') as f:            json.dump({'raw_text': result}, f)        print(f'[{i+1}] {img}: OK')        time.sleep(3)    except Exception as e:        print(f'[{i+1}] {img}: Error - {e}')print('Teste completo! Agora remove []:10 para processar todas.')

## 5. Fazer download dos resultadosGera `htr_output.tar.gz` para fazer download:

In [ ]:
import tarfilefrom google.colab import fileswith tarfile.open('htr_output.tar.gz', 'w:gz') as tar:    tar.add('htr_output')files.download('htr_output.tar.gz')print('Download pronto!')